# MouseTrap Scout — Edge Classifier Training

Trains a tiny int8 MobileNet v2 α=0.35 @ 96×96 classifier that runs on the XIAO ESP32-S3 Sense. Three classes: `rodent`, `person_or_pet`, `other`.

**Runtime requirements:** Colab free tier (T4 GPU recommended). Expect ~20–30 min total.

## Data sources
- **rodent** — iNaturalist observations of Order Rodentia (CC-licensed research-grade photos)
- **person_or_pet** — COCO 2017 val split, filtered to `person`, `cat`, `dog` categories
- **other** — COCO 2017 val split, filtered to images with no animals and no people (indoor scenes, objects, furniture)

All three sources are downloaded automatically — no manual labeling, no API keys needed beyond iNaturalist's public API.

## 0. Setup
Install TF 2.15+ (Colab default is usually fine) and verify GPU is available.

In [ ]:
import tensorflow as tf
import numpy as np
import os, json, shutil, hashlib, urllib.request, zipfile, tarfile, io, time, random
from pathlib import Path

print('TF version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)
print('Devices:', tf.config.list_physical_devices())

# Seed for reproducibility
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_ROOT = Path('/content/mousetrap_data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for c in ['rodent', 'person_or_pet', 'other']:
    (DATA_ROOT / c).mkdir(parents=True, exist_ok=True)

print('Data root:', DATA_ROOT)

## 1. Download rodent images from iNaturalist

Uses the iNaturalist public API. We query for **research-grade** observations in Order Rodentia that have CC-licensed photos. Target: ~2000 rodent images.

No API key required. Rate limited to ~1 req/sec to be polite.

In [ ]:
import requests

RODENTIA_TAXON_ID = 43094  # Class Mammalia → Order Rodentia on iNaturalist
TARGET_RODENT_COUNT = 2000
RODENT_DIR = DATA_ROOT / 'rodent'

def fetch_inat_observations(taxon_id, per_page=200, max_count=2000):
    """Yield (obs_id, photo_url) pairs from iNat API."""
    page = 1
    count = 0
    while count < max_count:
        url = (
            'https://api.inaturalist.org/v1/observations'
            f'?taxon_id={taxon_id}&photos=true&quality_grade=research'
            f'&license=cc-by,cc-by-nc,cc0&per_page={per_page}&page={page}'
            '&order_by=random'
        )
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            print(f'iNat API error {r.status_code} on page {page}')
            break
        results = r.json().get('results', [])
        if not results:
            break
        for obs in results:
            for photo in obs.get('photos', []):
                url_med = photo.get('url', '').replace('square', 'medium')
                if url_med:
                    yield (obs['id'], url_med)
                    count += 1
                    if count >= max_count:
                        return
        page += 1
        time.sleep(1.0)  # Rate limit

def download_image(url, dest_path, timeout=15):
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200 and len(r.content) > 1000:
            dest_path.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False

# Skip if we already have enough
existing = len(list(RODENT_DIR.glob('*.jpg')))
if existing >= TARGET_RODENT_COUNT * 0.9:
    print(f'Already have {existing} rodent images, skipping download')
else:
    print(f'Downloading up to {TARGET_RODENT_COUNT} rodent images from iNaturalist...')
    downloaded = 0
    for obs_id, photo_url in fetch_inat_observations(RODENTIA_TAXON_ID, max_count=TARGET_RODENT_COUNT * 2):
        if downloaded >= TARGET_RODENT_COUNT:
            break
        dest = RODENT_DIR / f'inat_{obs_id}_{downloaded}.jpg'
        if dest.exists():
            continue
        if download_image(photo_url, dest):
            downloaded += 1
            if downloaded % 100 == 0:
                print(f'  {downloaded}/{TARGET_RODENT_COUNT}')
    print(f'Downloaded {downloaded} rodent images')

print(f'Total rodent images: {len(list(RODENT_DIR.glob("*.jpg")))}')

## 2. Download COCO 2017 val split

COCO val is ~1 GB and contains 5000 images with full bbox annotations. We'll use it as the source for both `person_or_pet` and `other` classes.

In [ ]:
COCO_DIR = Path('/content/coco2017')
COCO_DIR.mkdir(parents=True, exist_ok=True)

COCO_IMAGES_URL = 'http://images.cocodataset.org/zips/val2017.zip'
COCO_ANNOTATIONS_URL = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'

def download_and_extract(url, dest_dir, marker_file):
    if (dest_dir / marker_file).exists():
        print(f'{marker_file} already present, skipping {url}')
        return
    print(f'Downloading {url} ...')
    local = dest_dir / url.split('/')[-1]
    with urllib.request.urlopen(url) as resp, open(local, 'wb') as f:
        shutil.copyfileobj(resp, f)
    print(f'Extracting {local} ...')
    with zipfile.ZipFile(local) as z:
        z.extractall(dest_dir)
    local.unlink()

download_and_extract(COCO_IMAGES_URL, COCO_DIR, 'val2017')
download_and_extract(COCO_ANNOTATIONS_URL, COCO_DIR, 'annotations')

print('COCO val images:', len(list((COCO_DIR / 'val2017').glob('*.jpg'))))

## 3. Split COCO into person_or_pet vs other

Read the COCO annotations JSON. For each image:
- If it contains `person`, `cat`, or `dog` → copy to `person_or_pet/`
- If it contains none of [person, cat, dog, mouse, squirrel, bird, horse, sheep, cow, bear] → copy to `other/`
- Otherwise skip (animal but not in our target set, to avoid confusing the model)

COCO does not have separate mouse/rodent labels, so any image with a cat or dog or person becomes a negative class example.

In [ ]:
ANN_PATH = COCO_DIR / 'annotations' / 'instances_val2017.json'
with open(ANN_PATH) as f:
    coco = json.load(f)

# Build lookup: image_id -> set of category names
cat_by_id = {c['id']: c['name'] for c in coco['categories']}
img_by_id = {img['id']: img['file_name'] for img in coco['images']}

from collections import defaultdict
img_categories = defaultdict(set)
for ann in coco['annotations']:
    img_categories[ann['image_id']].add(cat_by_id[ann['category_id']])

TARGET_PET_PERSON = {'person', 'cat', 'dog'}
ANY_ANIMAL = {'person', 'cat', 'dog', 'bird', 'horse', 'sheep', 'cow', 'bear',
              'elephant', 'giraffe', 'zebra'}

pet_person_files = []
other_files = []

for img_id, categories in img_categories.items():
    fname = img_by_id.get(img_id)
    if not fname:
        continue
    path = COCO_DIR / 'val2017' / fname
    if not path.exists():
        continue
    if categories & TARGET_PET_PERSON:
        pet_person_files.append(path)
    elif not (categories & ANY_ANIMAL):
        other_files.append(path)

# Also add images with NO annotations as 'other' (pure background)
annotated_ids = set(img_categories.keys())
for img in coco['images']:
    if img['id'] not in annotated_ids:
        path = COCO_DIR / 'val2017' / img['file_name']
        if path.exists():
            other_files.append(path)

print(f'COCO person_or_pet candidates: {len(pet_person_files)}')
print(f'COCO other candidates: {len(other_files)}')

# Copy to DATA_ROOT
PP_DIR = DATA_ROOT / 'person_or_pet'
OTHER_DIR = DATA_ROOT / 'other'

random.shuffle(pet_person_files)
random.shuffle(other_files)

TARGET_PP = 2000
TARGET_OTHER = 2000

for i, src in enumerate(pet_person_files[:TARGET_PP]):
    dst = PP_DIR / f'coco_pp_{i}.jpg'
    if not dst.exists():
        shutil.copy(src, dst)

for i, src in enumerate(other_files[:TARGET_OTHER]):
    dst = OTHER_DIR / f'coco_other_{i}.jpg'
    if not dst.exists():
        shutil.copy(src, dst)

print(f'Final counts:')
print(f'  rodent: {len(list((DATA_ROOT / "rodent").glob("*.jpg")))}')
print(f'  person_or_pet: {len(list((DATA_ROOT / "person_or_pet").glob("*.jpg")))}')
print(f'  other: {len(list((DATA_ROOT / "other").glob("*.jpg")))}')

## 4. Build tf.data pipeline

96×96 RGB, standard augmentation (flip, rotation, zoom, brightness). 80/20 train/val split.

In [ ]:
IMG_SIZE = 96
BATCH_SIZE = 64
CLASS_NAMES = ['rodent', 'person_or_pet', 'other']
NUM_CLASSES = len(CLASS_NAMES)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES,
    shuffle=True,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES,
    shuffle=False,
)

AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

train_ds = train_ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

print('Classes:', CLASS_NAMES)
print('Train batches:', tf.data.experimental.cardinality(train_ds).numpy())
print('Val batches:', tf.data.experimental.cardinality(val_ds).numpy())

## 5. Build MobileNet v2 α=0.35 with transfer learning

Start from ImageNet-pretrained weights. Freeze base, train head first, then fine-tune top layers.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    alpha=0.35,
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 6. Train — phase 1 (frozen base, head only)

In [ ]:
HEAD_EPOCHS = 10
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    verbose=1,
)

## 7. Train — phase 2 (unfreeze top, low lr fine-tune)

In [ ]:
base_model.trainable = True
# Freeze first ~100 layers, train the rest
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

FINE_EPOCHS = 15
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_EPOCHS,
    verbose=1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=4, restore_best_weights=True
        ),
    ],
)

val_loss, val_acc = model.evaluate(val_ds)
print(f'\nFinal val accuracy: {val_acc:.4f}  val loss: {val_loss:.4f}')

## 8. Post-training int8 quantization

Uses a representative dataset of 200 real images for calibration. Input and output are quantized to int8 for the smallest model and fastest ESP-NN kernels.

In [ ]:
def representative_dataset():
    # Pull 200 images from the val set
    count = 0
    for images, _ in val_ds.unbatch().batch(1).take(200):
        yield [tf.cast(images, tf.float32).numpy()]
        count += 1
        if count >= 200:
            return

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

OUT_DIR = Path('/content/output')
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'model.tflite').write_bytes(tflite_model)
(OUT_DIR / 'labels.txt').write_text('\n'.join(CLASS_NAMES) + '\n')

print(f'Quantized model size: {len(tflite_model) / 1024:.1f} KB')
print(f'Written to: {OUT_DIR / "model.tflite"}')

## 9. Verify quantized model accuracy

Run int8 inference on the val set to confirm quantization didn't destroy accuracy.

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print('Input:', input_details)
print('Output:', output_details)

in_scale, in_zp = input_details['quantization']
out_scale, out_zp = output_details['quantization']

correct = 0
total = 0
per_class_correct = [0] * NUM_CLASSES
per_class_total = [0] * NUM_CLASSES
confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

for images, labels in val_ds.unbatch().batch(1):
    img_float = tf.cast(images, tf.float32).numpy()
    img_int8 = (img_float / in_scale + in_zp).astype(np.int8)
    interpreter.set_tensor(input_details['index'], img_int8)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details['index'])
    pred = int(np.argmax(out[0]))
    true = int(labels[0])
    confusion[true][pred] += 1
    per_class_total[true] += 1
    if pred == true:
        correct += 1
        per_class_correct[true] += 1
    total += 1

print(f'\nQuantized val accuracy: {correct/total:.4f}  ({correct}/{total})')
print('\nPer-class accuracy:')
for i, name in enumerate(CLASS_NAMES):
    if per_class_total[i] > 0:
        print(f'  {name}: {per_class_correct[i]/per_class_total[i]:.4f}  ({per_class_correct[i]}/{per_class_total[i]})')

print('\nConfusion matrix (rows=truth, cols=pred):')
print('         ' + '  '.join(f'{c:>13}' for c in CLASS_NAMES))
for i, name in enumerate(CLASS_NAMES):
    row = '  '.join(f'{v:>13}' for v in confusion[i])
    print(f'  {name:>7}  {row}')

## 10. Download `model.tflite` + `labels.txt`

Run this cell and the files will appear in your browser downloads. Copy them into the repo:

```
scout_arduino/ml/model.tflite
scout_arduino/ml/labels.txt
```

Then rebuild the scout LittleFS image with `make build-fs` and upload.

In [ ]:
from google.colab import files
files.download(str(OUT_DIR / 'model.tflite'))
files.download(str(OUT_DIR / 'labels.txt'))
print('Done. Check your browser downloads folder.')

## 11. Optional: retrain with real scout data

Once the scout is running v1 and `Server/image_storage/` has accumulated real motion events, use this cell to re-train with real data mixed in.

Run `Server/scripts/export-training-data.sh` on the Mac to produce a `training-data-*.tar.gz`, then upload it here.

```python
from google.colab import files
uploaded = files.upload()  # Select the tarball
# Extract into DATA_ROOT, then rerun cells 4 onward
```